# GNN3: FPL + GNN Residual Surrogate (IEEE 34 Bus)

This notebook is a **thin Colab front-end** that:

1. Installs dependencies (OpenDSSDirect, PyTorch, PyTorch Geometric).
2. Clones / updates the `GNN-Sandia` repository and switches into it.
3. Calls external Python modules in `fpl_gnn/` to:
   - build the FPL linearization,
   - generate an FPL+residual dataset,
   - train a PFIdentityGNN on residuals.

All heavy logic lives in:
- `fpl_gnn/fpl_gnn_dataset.py`
- `fpl_gnn/fpl_gnn_train.py`.

In [ ]:
# 1) Core Python deps + OpenDSSDirect
!pip install -q pandas matplotlib
!pip install -q git+https://github.com/dss-extensions/OpenDSSDirect.py.git@master

# 2) Torch + PyTorch Geometric (for Colab CUDA 12.1 runtime)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-2.2.0+cu121.html

In [ ]:
# 3) Colab setup: clone repo and cd to script directory
import os

REPO_DIR = "/content/GNN-Sandia"
REPO_URL = "https://github.com/alitasavori/GNN-Sandia.git"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main
%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

In [ ]:
# 4) Imports from the FPL+GNN package inside the repo
import torch
import numpy as np
import pandas as pd

from fpl_gnn.fpl_gnn_dataset import generate_fpl_residual_dataset
from fpl_gnn.fpl_gnn_train import train_fpl_gnn

In [ ]:
# 5) Generate dataset and train FPL+GNN residual model
df_s, df_n, edge_path = generate_fpl_residual_dataset()

model_fpl_gnn = train_fpl_gnn(
    h_dim=96,
    num_layers=3,
    batch_size=64,
    epochs=50,
    test_frac=0.2,
)

print("Training complete.")